In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

In [2]:
trained_models = joblib.load("../models/trained_models.pkl")
top_models = joblib.load("../models/top_models.pkl")

In [3]:
category_cols = ["category", "day_of_week"]
target_col = "late_duration_min"

X_df = pd.DataFrame({
    "day_of_week": [4],
    "distance_km": [10.5],
    "category": ["dinner/drinks"]
})

X_df

,day_of_week,distance_km,category
0,4,10.5,dinner/drinks


In [5]:
def Cat_LabelEncoding(df, cols):
    modified_df = df.copy()

    le = LabelEncoder()

    for col in cols:
        modified_df[col] = le.fit_transform(modified_df[col])

    modified_df.drop(columns=cols)

    return modified_df

def Cat_OneHotEncoding(df, cols):
    modified_df = pd.get_dummies(df, columns=["category", "day_of_week"])

    return modified_df

def ensemble_predict(trained_models, model_names, X_df):
    preds = []

    for name in model_names:
        model_info = trained_models[name]

        if model_info["type"] == "linear":
            preds.append(
                model_info["model"].predict(
                    Cat_OneHotEncoding(X_df, category_cols)
                )
            )
        else:
            preds.append(
                model_info["model"].predict(
                    Cat_LabelEncoding(X_df, category_cols)
                )
            )

    return np.mean(preds, axis=0)

result_df = X_df.copy()

result_df[f"pred_{target_col}"] = ensemble_predict(
    trained_models,
    top_models,
    X_df
)

result_df["models_used"] = ", ".join(top_models)

result_df


,day_of_week,distance_km,category,pred_late_duration_min,models_used
0,4,10.5,dinner/drinks,38.531528,"gboost, random_forest"
